# 2장 2강: 신경망 최적화 — 실습문제 풀이

> **풀이 기준**: 프로젝트에 업로드된 `concrete_data.csv`로 코드를 실제 실행한 결과입니다. 출력만 반올림하고 계산에는 반올림 전 값을 사용했습니다.
>
> **가정한 내용**: 별도 판단이 필요한 부분은 없었습니다. 문제의 초기값·학습률·beta·eps를 그대로 사용했습니다.

---

## 실습 준비 (제공 코드)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("concrete_data.csv")
df.columns = df.columns.str.strip()
x = df["cement"].to_numpy()[0] / 100.0
y = df["concrete_compressive_strength"].to_numpy()[0]
theta0 = np.array([0.5, 0.1, 20.0, 20.0])

print("전체 데이터 크기:", df.shape)
print("첫 샘플의 계산용 입력 / 실제 정답:", x, y)
print("파라미터 순서: [w1, b1, w2, b2]")
print("초기 파라미터:", theta0)

전체 데이터 크기: (1030, 9)
첫 샘플의 계산용 입력 / 실제 정답: 5.4 79.99
파라미터 순서: [w1, b1, w2, b2]
초기 파라미터: [ 0.5  0.1 20.  20. ]


In [2]:
def forward_and_grad(theta, x_value, y_value):
    w1, b1, w2, b2 = theta
    z1 = w1 * x_value + b1
    h1 = np.maximum(0.0, z1)
    prediction = w2 * h1 + b2
    error = prediction - y_value
    loss = 0.5 * error ** 2
    relu_slope = 1.0 if z1 > 0 else 0.0
    grad_z1 = error * w2 * relu_slope
    gradients = np.array([
        grad_z1 * x_value,
        grad_z1,
        error * h1,
        error
    ])
    return prediction, loss, gradients

pred0, loss0, g0 = forward_and_grad(theta0, x, y)
print("초기 출력 / 손실:", round(pred0, 8), round(loss0, 8))
print("초기 기울기:", np.round(g0, 8))

초기 출력 / 손실: 76.0 7.96005
초기 기울기: [-430.92   -79.8    -11.172   -3.99 ]


---

## 실습 1. SGD와 Learning Rate 이해하기

### 문제 1-1. 같은 기울기에서 학습률만 바꾸기

In [3]:
print("theta0:", theta0)
print("g0    :", np.round(g0, 6))

theta_small = theta0 - 0.00001 * g0
theta_mid = theta0 - 0.0001 * g0
theta_large = theta0 - 0.001 * g0

pred_small, loss_small, _ = forward_and_grad(theta_small, x, y)
pred_mid, loss_mid, _ = forward_and_grad(theta_mid, x, y)
pred_large, loss_large, _ = forward_and_grad(theta_large, x, y)

for name, lr_value, th, p, l in [("small", 0.00001, theta_small, pred_small, loss_small),
                                  ("mid", 0.0001, theta_mid, pred_mid, loss_mid),
                                  ("large", 0.001, theta_large, pred_large, loss_large)]:
    print(f"[{name}] lr={lr_value}  theta={np.round(th, 6)}  출력={p:.6f}  손실={l:.6f}")

theta0: [ 0.5  0.1 20.  20. ]
g0    : [-430.92   -79.8    -11.172   -3.99 ]
[small] lr=1e-05  theta=[ 0.504309  0.100798 20.000112 20.00004 ]  출력=76.481709  손실=6.154053
[mid] lr=0.0001  theta=[ 0.543092  0.10798  20.001117 20.000399]  출력=80.817332  손실=0.342239
[large] lr=0.001  theta=[ 0.93092   0.1798   20.011172 20.00399 ]  출력=124.197520  손실=977.152413


#### 답변

**수식 및 계산 과정 (w1, 기울기 −430.92)**

| 학습률 | 수정량 lr × g | 새 w1 = 0.5 − lr × g |
|---|---:|---:|
| 0.00001 | −0.0043092 | 0.5 + 0.0043092 = **0.5043092** |
| 0.0001 | −0.043092 | 0.5 + 0.043092 = **0.543092** |
| 0.001 | −0.43092 | 0.5 + 0.43092 = **0.93092** |

**Q1. Optimizer와 역전파는 각각 무엇을 하나요?**

- 역전파: 현재 파라미터에서 손실에 대한 각 파라미터의 **기울기를 계산**합니다(이번에는 `forward_and_grad` 안에서 수행).
- Optimizer: 그 기울기를 받아 정해진 규칙(SGD, Adam 등)으로 **파라미터의 새 값을 계산**합니다.

**Q2. w1의 기울기 −430.92에 세 학습률을 적용하면 새 w1은 각각 얼마인가요?**

- 0.5043092, 0.543092, 0.93092입니다(위 표). 학습률이 10배 커질 때마다 수정량도 10배 커집니다.

**Q3. 세 학습률의 수정 후 출력과 손실은 각각 얼마인가요? 이 예시에서 가장 작은 손실은 어느 경우인가요?**

| 학습률 | 출력 | 손실 |
|---|---:|---:|
| 0.00001 | 76.481709 | 6.154053 |
| 0.0001 | 80.817332 | 0.342239 |
| 0.001 | 124.197520 | 977.152413 |

- 이 예시에서는 **lr = 0.0001**의 손실이 가장 작습니다.

**Q4. 학습률이 크면 항상 더 빨리 좋아지나요? 세 경우를 같은 초기값에서 비교하는 이유는 무엇인가요?**

- 아닙니다. lr = 0.001에서는 너무 크게 움직여 출력이 정답(79.99)을 크게 넘어 124.2가 되었고, 손실이 7.96에서 977.15로 **오히려 커졌습니다**. 학습률이 너무 작으면(0.00001) 개선이 느립니다.
- 같은 초기값과 같은 기울기에서 출발해야 결과 차이를 **학습률 하나의 효과**로 해석할 수 있기 때문입니다.

---

## 실습 2. Adam의 기록과 SGD와의 차이 확인하기

### 문제 2-1. Adam의 첫 번째 업데이트

In [4]:
lr = 0.0001
beta1 = 0.9
beta2 = 0.999
eps = 1e-8
t = 1
m0 = np.zeros_like(theta0)
v0 = np.zeros_like(theta0)

m1 = beta1 * m0 + (1 - beta1) * g0
v1 = beta2 * v0 + (1 - beta2) * g0 ** 2
m_hat1 = m1 / (1 - beta1 ** t)
v_hat1 = v1 / (1 - beta2 ** t)
adam_step1 = lr * m_hat1 / (np.sqrt(v_hat1) + eps)
theta_adam1 = theta0 - adam_step1

pred_adam1, loss_adam1, g_adam2 = forward_and_grad(theta_adam1, x, y)

print("m1        :", np.round(m1, 6))
print("v1        :", np.round(v1, 6))
print("m_hat1    :", np.round(m_hat1, 6))
print("v_hat1    :", np.round(v_hat1, 6))
print("adam_step1:", adam_step1)
print("theta_adam1:", np.round(theta_adam1, 6))
print("출력 / 손실:", round(pred_adam1, 6), round(loss_adam1, 6))

m1        : [-43.092   -7.98    -1.1172  -0.399 ]
v1        : [1.85692046e+02 6.36804000e+00 1.24814000e-01 1.59200000e-02]
m_hat1    : [-430.92   -79.8    -11.172   -3.99 ]
v_hat1    : [1.85692046e+05 6.36804000e+03 1.24813584e+02 1.59201000e+01]
adam_step1: [-1.00000000e-04 -1.00000000e-04 -9.99999999e-05 -9.99999997e-05]
theta_adam1: [ 0.5001  0.1001 20.0001 20.0001]
출력 / 손실: 76.01318 7.907548


#### 답변

**수식 및 계산 과정 (w1, g = −430.92)**

- m1 = 0.9 × 0 + 0.1 × (−430.92) = **−43.092**
- v1 = 0.999 × 0 + 0.001 × (−430.92)² = 0.001 × 185,692.0464 = **185.692046**
- m̂1 = −43.092 / (1 − 0.9¹) = −43.092 / 0.1 = **−430.92**
- v̂1 = 185.692046 / (1 − 0.999¹) = 185.692046 / 0.001 = **185,692.0464**
- √v̂1 = 430.92 → adam_step = 0.0001 × (−430.92) / (430.92 + 0.00000001) ≈ **−0.0001**
- 새 w1 = 0.5 − (−0.0001) = **0.5001**

**Q1. w1에 해당하는 m1과 v1, m_hat1과 v_hat1은 각각 얼마인가요?**

- m1 = −43.092, v1 ≈ 185.692046, m̂1 = −430.92, v̂1 ≈ 185,692.0464입니다.

**Q2. 첫 단계에서 네 파라미터가 각각 약 0.0001 증가한 이유는 무엇인가요? Adam이 언제나 일정한 크기로만 수정한다는 뜻인가요?**

- 첫 단계에서는 보정 후 m̂ = g, v̂ = g²가 되므로 m̂/√v̂ = g/|g| = **기울기의 부호(±1)**만 남습니다. 기울기가 모두 음수였으므로 네 파라미터가 모두 lr = 0.0001만큼 증가했습니다. 기울기 크기(−430.92 vs −3.99)와 관계없이 같은 크기로 움직인 것입니다.
- 언제나 그런 것은 아닙니다. 두 번째 단계부터는 m과 v에 과거 기록이 섞이므로, 기울기 방향이 계속 같은지·크기가 어떻게 변하는지에 따라 수정량이 달라집니다.

**Q3. m과 v는 무엇을 기록하고, 초기값 보정은 모델의 편향 b1·b2와 어떻게 다른가요?**

- m은 기울기의 이동평균(방향과 크기), v는 기울기 제곱의 이동평균(기울기가 얼마나 컸는지)입니다.
- 초기값 보정은 m·v가 0에서 시작해 초반에 0 쪽으로 치우치는 것을 (1 − βᵗ)로 나눠 바로잡는 계산입니다. 모델의 편향 b1·b2는 학습되는 파라미터로, 이름만 비슷할 뿐 전혀 다른 개념입니다. 보정은 b1·b2만이 아니라 네 파라미터 모두의 m·v에 적용됩니다.

**Q4. Adam도 Learning Rate가 필요한가요? eps는 어디에 더하나요?**

- 필요합니다. 이번에도 lr = 0.0001이 수정량의 크기를 결정했습니다.
- eps는 **√v̂를 구한 뒤 분모에** 더합니다: m̂ / (√v̂ + eps). 분모가 0이 되는 것을 막기 위해서입니다.

---

### 문제 2-2. 기록을 유지한 두 번째 업데이트와 SGD 비교

In [5]:
t = 2

# Adam 두 번째 업데이트 (theta_adam1, m1, v1, g_adam2 이어 사용)
m2 = beta1 * m1 + (1 - beta1) * g_adam2
v2 = beta2 * v1 + (1 - beta2) * (g_adam2 ** 2)
m_hat2 = m2 / (1 - beta1 ** t)
v_hat2 = v2 / (1 - beta2 ** t)
adam_step2 = lr * m_hat2 / (np.sqrt(v_hat2) + eps)
theta_adam2 = theta_adam1 - adam_step2
pred_adam2, loss_adam2, _ = forward_and_grad(theta_adam2, x, y)

# SGD 두 번 업데이트
theta_sgd1 = theta0 - lr * g0
pred_sgd1, loss_sgd1, g_sgd2 = forward_and_grad(theta_sgd1, x, y)
theta_sgd2 = theta_sgd1 - lr * g_sgd2
pred_sgd2, loss_sgd2, _ = forward_and_grad(theta_sgd2, x, y)

print("[Adam] 두 번째 기울기 g_adam2:", np.round(g_adam2, 6))
print("[Adam] m2:", np.round(m2, 6))
print("[Adam] v2:", np.round(v2, 6))
print("[Adam] 1회 손실:", round(loss_adam1, 6), " 2회 손실:", round(loss_adam2, 6))
print("[Adam] 최종 파라미터:", np.round(theta_adam2, 6), " 출력:", round(pred_adam2, 6))
print("[SGD ] 두 번째 기울기 g_sgd2:", np.round(g_sgd2, 6))
print("[SGD ] 1회 손실:", round(loss_sgd1, 6), " 2회 손실:", round(loss_sgd2, 6))
print("[SGD ] 최종 파라미터:", np.round(theta_sgd2, 6), " 출력:", round(pred_sgd2, 6))

[Adam] 두 번째 기울기 g_adam2: [-429.498701  -79.536796  -11.137641   -3.97682 ]
[Adam] m2: [-81.73267  -15.13568   -2.119244  -0.756782]
[Adam] v2: [3.69975488e+02 1.26877740e+01 2.48736000e-01 3.17190000e-02]
[Adam] 1회 손실: 7.907548  2회 손실: 7.855225
[Adam] 최종 파라미터: [ 0.5002  0.1002 20.0002 20.0002]  출력: 76.026359
[SGD ] 두 번째 기울기 g_sgd2: [89.356852 16.547565  2.515649  0.827332]
[SGD ] 1회 손실: 0.342239  2회 손실: 0.014742
[SGD ] 최종 파라미터: [ 0.534156  0.106325 20.000866 20.000316]  출력: 79.818292


#### 답변

**결과 정리**

| 방식 | 1회 후 손실 | 2회 후 손실 | 2회 후 출력 |
|---|---:|---:|---:|
| SGD | 0.342239 | **0.014742** | 79.818292 |
| Adam | 7.907548 | 7.855225 | 76.026359 |

**Q1. 두 번째 Adam 단계에서 m·v를 0으로 다시 만들면 무엇이 달라지나요? t는 얼마인가요?**

- t = **2**입니다.
- m·v를 0으로 되돌리면 첫 단계의 기울기 기록이 사라지고, t = 1처럼 계산이 다시 시작됩니다. 그러면 Adam이 과거 기울기를 반영한다는 핵심 특징이 사라지고, 매번 "부호 × lr" 크기로만 움직이는 계산이 됩니다. 또 t = 2로 보정하면 0에서 시작한 m·v와 보정 분모가 맞지 않아 수정량이 잘못 계산됩니다.

**Q2. 왜 두 번째 업데이트의 기울기를 새로 계산해야 하나요? SGD와 Adam이 두 번째에 같은 기울기를 사용하나요?**

- 파라미터가 바뀌면 출력과 오차가 달라지므로 기울기도 달라집니다. 기울기는 **현재 파라미터 위치에서의 방향**이므로 매번 새로 구해야 합니다.
- 같은 기울기를 쓰지 않습니다. SGD는 θ_sgd1에서 구한 g_sgd2(w1 기울기 약 **+89.36**, 이미 정답을 넘어서 양수), Adam은 θ_adam1에서 구한 g_adam2(w1 기울기 약 **−429.50**, 아직 정답보다 작아서 음수)를 사용합니다. 첫 단계 후 위치가 서로 달랐기 때문입니다.

**Q3. 두 번 업데이트한 후 SGD와 Adam의 손실은 각각 얼마인가요? 이것만으로 SGD가 언제나 더 좋다고 결론 내릴 수 있나요?**

- SGD **0.014742**, Adam **7.855225**입니다.
- 결론 내릴 수 없습니다. 이번 예는 샘플 하나, 2회 업데이트, 같은 lr = 0.0001이라는 특정 조건입니다. 이 문제는 기울기가 커서(−430) SGD가 lr × g로 크게 움직였고, Adam은 첫 단계에서 약 0.0001씩만 움직였습니다. Adam에 맞는 학습률을 쓰거나, 여러 샘플·많은 반복으로 비교하면 결과가 달라질 수 있습니다.

**Q4. 기본 SGD와 Adam은 현재 기울기와 과거 기록을 어떻게 다르게 사용하나요?**

- 기본 SGD: **현재 기울기만** 사용하며, 수정량 = lr × g로 기울기 크기에 비례합니다. 과거 기록이 없습니다.
- Adam: 현재 기울기를 **과거 기록 m(기울기 평균)·v(기울기 제곱 평균)에 섞어** 갱신하고, m̂/√v̂로 파라미터마다 수정 크기를 자동 조절합니다.

---

## 과제. 다른 샘플에서 두 Optimizer의 한 단계 복습

### 문제 3-1. 세 번째 샘플에서 SGD와 Adam 각각 한 번 적용

In [6]:
task_x = df["cement"].to_numpy()[2] / 100.0
task_y = df["concrete_compressive_strength"].to_numpy()[2]
task_theta0 = np.array([0.5, 0.1, 20.0, 20.0])
task_lr = 0.0001
task_beta1 = 0.9
task_beta2 = 0.999
task_eps = 1e-8
task_t = 1
task_m0 = np.zeros_like(task_theta0)
task_v0 = np.zeros_like(task_theta0)

task_pred0, task_loss0, task_g = forward_and_grad(task_theta0, task_x, task_y)
print("초기 출력 / 손실:", round(task_pred0, 6), round(task_loss0, 6))
print("초기 기울기:", np.round(task_g, 6))

# SGD
task_theta_sgd = task_theta0 - task_lr * task_g
task_pred_sgd, task_loss_sgd, _ = forward_and_grad(task_theta_sgd, task_x, task_y)

# Adam
task_m = task_beta1 * task_m0 + (1 - task_beta1) * task_g
task_v = task_beta2 * task_v0 + (1 - task_beta2) * task_g ** 2
task_m_hat = task_m / (1 - task_beta1 ** task_t)
task_v_hat = task_v / (1 - task_beta2 ** task_t)
task_theta_adam = task_theta0 - task_lr * task_m_hat / (np.sqrt(task_v_hat) + task_eps)
task_pred_adam, task_loss_adam, _ = forward_and_grad(task_theta_adam, task_x, task_y)

print("[SGD ] 새 파라미터:", np.round(task_theta_sgd, 6), " 출력:", round(task_pred_sgd, 6), " 손실:", round(task_loss_sgd, 6))
print("[Adam] m_hat:", np.round(task_m_hat, 6))
print("[Adam] v_hat:", np.round(task_v_hat, 6))
print("[Adam] 새 파라미터:", np.round(task_theta_adam, 6), " 출력:", round(task_pred_adam, 6), " 손실:", round(task_loss_adam, 6))
print("손실 비교  초기:", round(task_loss0, 6), " SGD:", round(task_loss_sgd, 6), " Adam:", round(task_loss_adam, 6))

초기 출력 / 손실: 55.25 112.2002
초기 기울기: [996.17    299.6      26.40225  14.98   ]
[SGD ] 새 파라미터: [ 0.400383  0.07004  19.99736  19.998502]  출력: 48.021072  손실: 30.039556
[Adam] m_hat: [996.17    299.6      26.40225  14.98   ]
[Adam] v_hat: [9.92354669e+05 8.97601600e+04 6.97078805e+02 2.24400400e+02]
[Adam] 새 파라미터: [ 0.4999  0.0999 19.9999 19.9999]  출력: 55.241074  손실: 112.066525
손실 비교  초기: 112.2002  SGD: 30.039556  Adam: 112.066525


#### 답변

**수식 및 계산 과정 (w1, g = 996.17)**

- SGD: 새 w1 = 0.5 − 0.0001 × 996.17 = 0.5 − 0.099617 = **0.400383**
- Adam: m = 0.1 × 996.17 = 99.617 → m̂ = 99.617 / 0.1 = **996.17**
  v = 0.001 × 996.17² = 992.354669 → v̂ = 992.354669 / 0.001 = **992,354.669**
  √v̂ = 996.17 → 수정량 = 0.0001 × 996.17 / (996.17 + 1e-8) ≈ 0.0001 → 새 w1 = **0.4999**

**Q1. 세 번째 샘플의 초기 출력·손실·기울기는 무엇인가요?**

- 출력 **55.25**, 손실 **112.2002**, 기울기 [w1, b1, w2, b2] = **[996.17, 299.6, 26.40225, 14.98]**입니다.

**Q2. SGD를 한 번 적용한 새 w1은 얼마이며, 수정 후 손실은 얼마인가요?**

- 새 w1 = **0.400383**, 수정 후 손실 = **30.039556**입니다(출력 48.021072).

**Q3. Adam 첫 단계에서 w1의 보정된 m·v는 무엇이며, 새 w1과 수정 후 손실은 얼마인가요?**

- m̂ = **996.17**, v̂ ≈ **992,354.669**이고, 새 w1 = **0.4999**, 수정 후 손실 ≈ **112.066525**(출력 55.241074)입니다.

**Q4. 같은 학습률인데 두 방식의 w1 수정량이 다른 이유는 무엇인가요? 두 방식 모두 증가하나요, 감소하나요?**

- SGD는 lr × g = 0.0001 × 996.17 ≈ 0.0996으로 기울기 크기에 비례해 움직이지만, Adam 첫 단계는 m̂/√v̂ ≈ 1이므로 lr = 0.0001만큼만 움직입니다. 그래서 SGD의 수정량이 약 996배 큽니다.
- 기울기가 양수이므로 두 방식 모두 w1을 **감소**시킵니다(0.5 → 0.400383, 0.5 → 0.4999).

**Q5. 이번 과제에서는 왜 Adam 기록을 초기화하나요? 다음 업데이트를 이어 한다면 무엇을 유지하고 무엇을 새로 계산해야 하나요?**

- 원래 θ0에서 다시 출발하는 **독립적인 새 실험**이기 때문에 파라미터·m·v·t를 모두 초기 조건으로 맞춥니다.
- 이어서 학습한다면 **유지**: 현재 파라미터(task_theta_adam), 보정 전 기록 m·v, 순번 t(→2로 증가). **새로 계산**: 현재 파라미터에서의 기울기, 갱신된 m·v, t = 2 기준의 보정값과 수정량입니다.

---

## 실습 마무리

**1. Optimizer가 직접 사용하는 정보와 수행하는 일을 설명하세요.**

- 현재 파라미터, 역전파로 구한 기울기, 학습률(Adam이라면 m·v·t·beta·eps 같은 기록과 설정)을 사용해 파라미터의 새 값을 계산합니다. 정답이나 손실 값을 파라미터에 직접 넣지는 않습니다.

**2. 이번 기본 SGD의 식을 말로 읽어 보세요.**

- "새 파라미터는 기존 파라미터에서 학습률 곱하기 기울기를 뺀 값이다." (θ_new = θ_old − lr × g)

**3. Adam에서 m과 v는 각각 무엇인가요?**

- m은 기울기의 이동평균(방향을 꾸준히 유지하는 정도), v는 기울기 제곱의 이동평균(기울기가 얼마나 컸는지)입니다.

**4. Adam을 쓰면 학습률을 설정하지 않아도 되나요?**

- 아닙니다. Adam도 lr이 수정량의 기본 크기를 정합니다. 실습 2-1에서 첫 수정량이 정확히 lr(0.0001) 크기였던 것이 그 증거입니다.

**5. 다음 업데이트에서 반드시 다시 계산해야 하는 것은 무엇인가요?**

- 바뀐 파라미터에서의 **출력·손실·기울기**입니다. Adam이라면 이 새 기울기로 m·v를 갱신하고, t를 1 늘린 보정값과 수정량도 다시 계산합니다.

**6. 이번 결과로 SGD와 Adam의 일반적인 우열을 정할 수 있나요?**

- 없습니다. 샘플 하나·적은 업데이트 횟수·같은 학습률이라는 좁은 조건에서의 결과입니다. 이번에는 SGD의 손실이 더 빨리 줄었지만, 데이터·모델·학습률·반복 횟수에 따라 결과는 달라질 수 있습니다.